In [1]:

import pandas as pd
from pathlib import Path
import zipfile

# Folder where you downloaded the datasets
data_folder = Path.home() / "Documents/stanford_policing_data"

# Find ZIP archives
files = sorted(data_folder.glob("*.csv.zip"))

print("Total datasets:", len(files))

for file in files:
    print(file.name)


Total datasets: 89
ar_little_rock_2020_04_01.csv.zip
az_gilbert_2020_04_01.csv.zip
az_mesa_2023_01_26.csv.zip
az_statewide_2020_04_01.csv.zip
ca_anaheim_2020_04_01.csv.zip
ca_bakersfield_2020_04_01.csv.zip
ca_long_beach_2020_04_01.csv.zip
ca_los_angeles_2020_04_01.csv.zip
ca_oakland_2020_04_01.csv.zip
ca_san_bernardino_2020_04_01.csv.zip
ca_san_diego_2020_04_01.csv.zip
ca_san_francisco_2020_04_01.csv.zip
ca_san_jose_2020_04_01.csv.zip
ca_santa_ana_2020_04_01.csv.zip
ca_statewide_2023_01_26.csv.zip
ca_stockton_2020_04_01.csv.zip
co_aurora_2023_01_26.csv.zip
co_denver_2020_04_01.csv.zip
co_statewide_2020_04_01.csv.zip
ct_hartford_2020_04_01.csv.zip
ct_statewide_2020_04_01.csv.zip
fl_saint_petersburg_2020_04_01.csv.zip
fl_statewide_2020_04_01.csv.zip
fl_tampa_2020_04_01.csv.zip
ga_statewide_2020_04_01.csv.zip
ia_statewide_2020_04_01.csv.zip
id_idaho_falls_2020_04_01.csv.zip
il_chicago_2023_01_26.csv.zip
il_statewide_2020_04_01.csv.zip
in_fort_wayne_2020_04_01.csv.zip
ks_wichita_2023_01_26

In [2]:

inventory = []

for file in files:
    try:
        with zipfile.ZipFile(file) as z:
            csv_names = [
                name for name in z.namelist()
                if name.lower().endswith(".csv")
            ]

            if not csv_names:
                continue

            with z.open(csv_names[0]) as f:
                sample = pd.read_csv(
                    f, nrows=1000, low_memory=False
                )

            inventory.append({
                "Dataset": file.name,
                "Size_MB": round(
                    file.stat().st_size / 1024**2, 2
                ),
                "Columns": len(sample.columns),
                "Sample_Rows": len(sample),
                "Column_Names": ", ".join(sample.columns)
            })

    except Exception as e:
        print("Error reading:", file.name, e)

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

inventory_df.to_csv(
    data_folder / "dataset_inventory.csv",
    index=False
)


Error reading: tx_statewide_2020_04_01.csv.zip File is not a zip file


,Dataset,Size_MB,Columns,Sample_Rows,Column_Names
0,ar_little_rock_2020_04_01.csv.zip,0.17,13,1000,"raw_row_number, date, time, lat, lng, subject_..."
1,az_gilbert_2020_04_01.csv.zip,8.51,12,1000,"raw_row_number, date, time, location, lat, lng..."
2,az_mesa_2023_01_26.csv.zip,4.29,20,1000,"raw_row_number, date, time, location, lat, lng..."
3,az_statewide_2020_04_01.csv.zip,61.80,29,1000,"raw_row_number, date, time, location, county_n..."
4,ca_anaheim_2020_04_01.csv.zip,0.24,4,1000,"raw_row_number, date, type, reason_for_stop"
...,...,...,...,...,...
83,wa_statewide_2020_04_01.csv.zip,239.03,30,1000,"raw_row_number, date, time, location, lat, lng..."
84,wa_tacoma_2020_04_01.csv.zip,5.30,15,1000,"raw_row_number, date, time, location, lat, lng..."
85,wi_madison_2023_01_26.csv.zip,7.72,25,1000,"raw_row_number, date, time, location, lat, lng..."
86,wi_statewide_2020_04_01.csv.zip,48.96,44,1000,"raw_row_number, date, time, location, lat, lng..."


In [3]:

states = ["az", "nc", "oh", "wa"]

for state in states:
    matches = [
        f for f in files
        if f.name.startswith(state + "_statewide")
    ]

    if not matches:
        print(f"\n{state.upper()}: Not found")
        continue

    file = matches[0]

    with zipfile.ZipFile(file) as z:
        csv_name = next(
            n for n in z.namelist()
            if n.lower().endswith(".csv")
        )

        with z.open(csv_name) as f:
            df = pd.read_csv(f, nrows=1000)

    print(f"\n{'=' * 40}")
    print("STATE:", state.upper())
    print("FILE:", file.name)
    print("NUMBER OF COLUMNS:", len(df.columns))
    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())
    print("\nFIRST FIVE RECORDS:")
    display(df.head())



STATE: AZ
FILE: az_statewide_2020_04_01.csv.zip
NUMBER OF COLUMNS: 29

COLUMN NAMES:
['raw_row_number', 'date', 'time', 'location', 'county_name', 'subject_race', 'subject_sex', 'officer_id_hash', 'type', 'violation', 'arrest_made', 'citation_issued', 'warning_issued', 'outcome', 'contraband_found', 'contraband_drugs', 'contraband_other', 'search_conducted', 'search_person', 'search_vehicle', 'search_basis', 'reason_for_stop', 'vehicle_type', 'vehicle_year', 'raw_Ethnicity', 'raw_OutcomeOfStop', 'raw_ReasonForStop', 'raw_TypeOfSearch', 'raw_ViolationsObserved']

FIRST FIVE RECORDS:


,raw_row_number,date,time,location,county_name,subject_race,subject_sex,officer_id_hash,type,violation,...,search_vehicle,search_basis,reason_for_stop,vehicle_type,vehicle_year,raw_Ethnicity,raw_OutcomeOfStop,raw_ReasonForStop,raw_TypeOfSearch,raw_ViolationsObserved
0,1,2009-01-06,01:00:00,I10 165,Maricopa County,hispanic,male,08a56c4f70,vehicular,other moving violation,...,False,NaN,Moving violation,12PU,2007.0,H,WA,H,NaN,OM
1,2,2009-01-18,01:58:00,US60 185,Maricopa County,white,male,08a56c4f70,vehicular,speed,...,False,NaN,Moving violation,4DSD,2004.0,W,WA,H,NaN,SP
2,3,2009-01-20,01:25:00,SR202 16,Maricopa County,hispanic,male,08a56c4f70,vehicular,NaN,...,False,NaN,Moving violation,4DSD,2001.0,H,WA,H,NaN,LU
3,4,2009-01-24,19:52:00,US60 172.4,Maricopa County,hispanic,female,08a56c4f70,vehicular,NaN,...,False,NaN,Moving violation,4DSD,2003.0,H,CI,H,NaN,LU
4,5,2009-01-25,02:25:00,US60 181,Maricopa County,white,female,08a56c4f70,vehicular,NaN,...,False,NaN,Moving violation,4DSD,2001.0,W,WA,H,NaN,LU



STATE: NC
FILE: nc_statewide_2020_04_01.csv.zip
NUMBER OF COLUMNS: 29

COLUMN NAMES:
['raw_row_number', 'date', 'time', 'location', 'county_name', 'subject_age', 'subject_race', 'subject_sex', 'officer_id_hash', 'department_name', 'type', 'arrest_made', 'citation_issued', 'warning_issued', 'outcome', 'contraband_found', 'contraband_drugs', 'contraband_weapons', 'frisk_performed', 'search_conducted', 'search_person', 'search_vehicle', 'search_basis', 'reason_for_frisk', 'reason_for_search', 'reason_for_stop', 'raw_Ethnicity', 'raw_Race', 'raw_action_description']

FIRST FIVE RECORDS:


,raw_row_number,date,time,location,county_name,subject_age,subject_race,subject_sex,officer_id_hash,department_name,...,search_conducted,search_person,search_vehicle,search_basis,reason_for_frisk,reason_for_search,reason_for_stop,raw_Ethnicity,raw_Race,raw_action_description
0,1,2000-05-22,21:10:00,Unknown,NaN,45,white,male,22e35044ed,"NC Division of Motor Vehicles, License and The...",...,False,False,False,NaN,NaN,NaN,Speed Limit Violation,N,W,Citation Issued
1,2,2000-01-03,07:52:00,Unknown,NaN,20,white,male,22e35044ed,"NC Division of Motor Vehicles, License and The...",...,False,False,False,NaN,NaN,NaN,Other Motor Vehicle Violation,N,W,Citation Issued
2,3,2000-01-06,10:30:00,Unknown,NaN,23,hispanic,male,22e35044ed,"NC Division of Motor Vehicles, License and The...",...,False,False,False,NaN,NaN,NaN,Speed Limit Violation,H,U,Written Warning
3,4,2000-01-06,14:50:00,Unknown,NaN,60,white,male,22e35044ed,"NC Division of Motor Vehicles, License and The...",...,False,False,False,NaN,NaN,NaN,Vehicle Regulatory Violation,N,W,Citation Issued
4,5,2000-01-06,14:50:00,Unknown,NaN,27,white,male,22e35044ed,"NC Division of Motor Vehicles, License and The...",...,False,False,False,NaN,NaN,NaN,Vehicle Regulatory Violation,N,W,Citation Issued



STATE: OH
FILE: oh_statewide_2020_04_01.csv.zip
NUMBER OF COLUMNS: 24

COLUMN NAMES:
['raw_row_number', 'date', 'time', 'location', 'lat', 'lng', 'county_name', 'subject_race', 'subject_sex', 'officer_id_hash', 'department_name', 'type', 'violation', 'arrest_made', 'warning_issued', 'outcome', 'contraband_found', 'contraband_drugs', 'search_conducted', 'search_basis', 'raw_DISP_STRING', 'raw_ORC_STRING', 'raw_DISPOSITIONS', 'raw_race']

FIRST FIVE RECORDS:


,raw_row_number,date,time,location,lat,lng,county_name,subject_race,subject_sex,officer_id_hash,...,warning_issued,outcome,contraband_found,contraband_drugs,search_conducted,search_basis,raw_DISP_STRING,raw_ORC_STRING,raw_DISPOSITIONS,raw_race
0,1,2010-01-01,00:01:02,SR821M10NOBR,39.7566,-81.5327,Noble County,white,male,dc4f689699,...,True,warning,NaN,NaN,False,NaN,NE -M WARN,NaN,NaN,white
1,2,2010-01-01,00:01:15,KENDALL ST&W MAIN ST,38.8672,-83.7901,Brown County,white,male,47d2f5ae1d,...,True,warning,NaN,NaN,False,NaN,NE -M WARN,NaN,NaN,white
2,3,2010-01-01,00:02:50,INDIAN RIPPLE RD&TONAWANDA TRL,39.6962,-84.1019,Greene County,black,male,1d6afeaa47,...,True,warning,NaN,NaN,False,NaN,ENF -M WARN,----.--D-,NaN,black
3,4,2010-01-01,00:03:20,BALDWIN BLVD&COOPER FOSTER PAR,41.4165,-82.1701,Lorain County,white,female,3918039772,...,True,warning,NaN,NaN,False,NaN,NE -F WARN,NaN,NaN,white
4,5,2010-01-01,00:04:10,EVERHARD RD NW&I77,40.8593,-81.4268,Stark County,black,male,3190f7fb63,...,True,warning,NaN,NaN,False,NaN,NE -M WARN,NaN,NaN,black



STATE: WA
FILE: wa_statewide_2020_04_01.csv.zip
NUMBER OF COLUMNS: 30

COLUMN NAMES:
['raw_row_number', 'date', 'time', 'location', 'lat', 'lng', 'county_name', 'subject_age', 'subject_race', 'subject_sex', 'officer_race', 'officer_sex', 'department_name', 'type', 'violation', 'arrest_made', 'citation_issued', 'warning_issued', 'outcome', 'contraband_found', 'frisk_performed', 'search_conducted', 'search_basis', 'raw_officer_race', 'raw_officer_gender', 'raw_contact_type', 'raw_driver_race', 'raw_driver_gender', 'raw_search_type', 'raw_enforcements']

FIRST FIVE RECORDS:


,raw_row_number,date,time,location,lat,lng,county_name,subject_age,subject_race,subject_sex,...,frisk_performed,search_conducted,search_basis,raw_officer_race,raw_officer_gender,raw_contact_type,raw_driver_race,raw_driver_gender,raw_search_type,raw_enforcements
0,1,2009-10-27,11:00:00,S-012-344,46.114221,-118.224510,Walla Walla County,30.0,white,male,...,False,False,NaN,WHITE,-,--,White,M,N,-|-|-
1,2,2009-10-04,21:00:00,I-090-10,47.580522,-122.170289,King County,29.0,black,male,...,False,False,NaN,WHITE,-,--,African American,M,N,-|-
2,3,2009-10-04,03:00:00,I-090-72,47.250003,-121.189883,Kittitas County,25.0,white,male,...,False,False,NaN,WHITE,-,--,White,M,N,NaN
3,4,2009-10-04,13:00:00,S-014-3,45.615344,-122.612893,Clark County,19.0,white,female,...,False,False,NaN,WHITE,-,--,White,F,N,-|-|-|-|-|-
4,5,2009-10-11,08:00:00,S-003-40,47.590283,-122.698671,Kitsap County,25.0,white,male,...,False,False,NaN,WHITE,-,--,White,M,N,-|-


In [5]:

import pandas as pd
import zipfile
from pathlib import Path

# Location of downloaded Stanford datasets
folder = Path.home() / "Documents/stanford_policing_data"

# Find Montana Statewide dataset
files = sorted(folder.glob("mt_statewide*.csv.zip"))

print("Files found:", [f.name for f in files])

if not files:
    raise FileNotFoundError("Montana dataset not found")

file_path = files[-1]

# Open ZIP without extracting it
with zipfile.ZipFile(file_path) as z:
    csv_name = next(
        name for name in z.namelist()
        if name.lower().endswith(".csv")
    )

    with z.open(csv_name) as f:
        df = pd.read_csv(f, nrows=10000, low_memory=False)

print("Dataset:", file_path.name)
print("Sample rows:", len(df))
print("Columns:", len(df.columns))

display(df.head(20))


Files found: ['mt_statewide_2023_01_26.csv.zip']
Dataset: mt_statewide_2023_01_26.csv.zip
Sample rows: 10000
Columns: 30


,raw_row_number,date,time,location,lat,lng,county_name,subject_age,subject_race,subject_sex,...,reason_for_stop,vehicle_make,vehicle_model,vehicle_type,vehicle_registration_state,vehicle_year,raw_Race,raw_Ethnicity,raw_SearchType,raw_search_basis
0,1,2009-01-01,02:10:53,US 89 N MM10 (SB),47.572738,-111.802932,Cascade County,16.0,white,female,...,--- - HIT AND RUN,FORD,EXPLORER,SPORT UTILITY,MT,1994.0,W,N,NO SEARCH REQUESTED,NaN
1,2,2009-01-02,11:34:19,HWY 93 SO AND ANNS LANE S/B,46.761225,-114.081142,Missoula County,19.0,white,male,...,EXPIRED TAG ( - MONTHS OR LESS ),GMC,TK,TRUCK,MT,1996.0,W,N,NO SEARCH REQUESTED,NaN
2,3,2009-01-03,11:36:42,P007 HWY 93 MM 77 N/B,46.694683,-114.073505,Missoula County,17.0,white,male,...,SPEED,GMC,YUKON,SPORT UTILITY,MT,1999.0,W,N,NO SEARCH REQUESTED,NaN
3,4,2009-01-04,10:33:11,P007 HWY 93 MM 81 S/B,46.727388,-114.079027,Missoula County,17.0,NaN,female,...,SPEED,HOND,CR-V,SPORT UTILITY,MT,2002.0,W,NaN,NaN,NaN
4,5,2009-01-04,10:46:43,P007 HWY 93 MM 81 N/B,46.729518,-114.079150,Missoula County,31.0,NaN,male,...,SPEED,TOYT,TERCEL,SEDAN,MT,1992.0,I,NaN,NaN,NaN
5,6,2009-01-04,14:41:57,P007 HWY 93 MM 67 S/B,46.532282,-114.110712,Ravalli County,20.0,NaN,male,...,OTHER TAG/REGISTRATION VIOLATIONS,JEEP,CHE,SPORT UTILITY,MT,1998.0,W,NaN,NaN,NaN
6,7,2009-01-04,17:45:40,WESTBOUND TRUCK SCALE,45.935742,-108.342832,Yellowstone County,30.0,white,male,...,--- - OTHER DEFECTIVE EQUIPMENT,SUBA,NaN,PK,MT,2006.0,W,N,NO SEARCH REQUESTED,NaN
7,8,2009-01-05,15:32:41,P007 HWY 93 MM 79 S/B,46.699178,-114.073923,Missoula County,34.0,NaN,female,...,SPEED,GMC,YUK,SPORT UTILITY,MT,2004.0,W,NaN,NaN,NaN
8,9,2009-01-06,16:45:12,INTERSECTION OF HWY 312 & SHEPHERD ROAD,45.902608,-108.342213,Yellowstone County,21.0,white,male,...,--- - SPEED OVER LEGAL,BMW,NaN,4D,MT,1992.0,W,N,NO SEARCH REQUESTED,NaN
9,10,2009-01-06,16:45:17,INTERSECTION OF HWY 312 & SHEPHERD ROAD,45.902597,-108.342210,Yellowstone County,18.0,NaN,female,...,NaN,HOND,NaN,4D,MT,1987.0,W,NaN,NaN,NaN


In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import zipfile
import re

data_folder = Path.home() / "Documents/stanford_policing_data"
output_folder = data_folder / "merged_by_state"
output_folder.mkdir(parents=True, exist_ok=True)

# Group ZIP files by state abbreviation.
files_by_state = defaultdict(list)

for file in sorted(data_folder.glob("*.csv.zip")):
    match = re.match(r"^([a-z]{2})[_\-.]", file.name, re.IGNORECASE)

    if not match:
        raise ValueError(f"Cannot identify state from filename: {file.name}")

    files_by_state[match.group(1).upper()].append(file)

# Process in chunks to avoid loading all datasets into memory.
for state, files in sorted(files_by_state.items()):
    output_file = output_folder / f"{state}_merged.csv"

    # Find the union of columns across all CSVs for this state.
    columns = []
    members_by_file = {}

    for file in files:
        with zipfile.ZipFile(file) as archive:
            members = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv") and not name.endswith("/")
            ]

            if not members:
                raise ValueError(f"No CSV found in {file.name}")

            members_by_file[file] = members

            for member in members:
                with archive.open(member) as csv_file:
                    header = pd.read_csv(csv_file, nrows=0).columns
                    for column in header:
                        if column not in columns:
                            columns.append(column)

    if "source_file" not in columns:
        columns.append("source_file")

    total_rows = 0

    with output_file.open("w", encoding="utf-8", newline="") as output:
        # Write the header once.
        pd.DataFrame(columns=columns).to_csv(output, index=False)

        for file in files:
            with zipfile.ZipFile(file) as archive:
                for member in members_by_file[file]:
                    with archive.open(member) as csv_file:
                        for chunk in pd.read_csv(
                            csv_file,
                            chunksize=100_000,
                            dtype=str,
                        ):
                            chunk["source_file"] = file.name
                            chunk.reindex(columns=columns).to_csv(
                                output,
                                index=False,
                                header=False,
                            )
                            total_rows += len(chunk)

    print(
        f"{state}: {len(files)} archives → "
        f"{total_rows:,} rows saved to {output_file.name}"
    )

print(f"\nSaved merged files in: {output_folder}")

AR: 1 archives → 13,641 rows saved to AR_merged.csv
AZ: 3 archives → 4,135,773 rows saved to AZ_merged.csv
CA: 12 archives → 39,593,160 rows saved to CA_merged.csv
CO: 3 archives → 5,241,190 rows saved to CO_merged.csv
CT: 2 archives → 1,193,778 rows saved to CT_merged.csv
FL: 3 archives → 10,132,161 rows saved to FL_merged.csv
GA: 1 archives → 1,906,772 rows saved to GA_merged.csv
IA: 1 archives → 2,441,335 rows saved to IA_merged.csv
ID: 1 archives → 104,814 rows saved to ID_merged.csv
IL: 2 archives → 14,856,271 rows saved to IL_merged.csv
IN: 1 archives → 265,019 rows saved to IN_merged.csv
KS: 1 archives → 1,030,376 rows saved to KS_merged.csv
KY: 2 archives → 153,483 rows saved to KY_merged.csv
LA: 1 archives → 512,092 rows saved to LA_merged.csv
MA: 1 archives → 3,416,238 rows saved to MA_merged.csv
MD: 2 archives → 4,524,424 rows saved to MD_merged.csv
MI: 1 archives → 800,302 rows saved to MI_merged.csv
MN: 1 archives → 675,156 rows saved to MN_merged.csv
MO: 1 archives → 10,0

In [2]:
import pandas as pd
from pathlib import Path

data_folder = (
    Path.home()
    / "Documents/stanford_policing_data/merged_by_state"
)

files = sorted(data_folder.glob("*_merged.csv"))

print(f"Total datasets: {len(files)}")

for file in files:
    df = pd.read_csv(file, nrows=1000, low_memory=False)

    print("\n" + "=" * 80)
    print(f"DATASET: {file.name}")
    print(f"Columns: {len(df.columns)}")
    print(f"Rows inspected: {len(df):,} (maximum 1,000)")

    print("\nFIRST 5 ROWS:")
    print(df.head().to_string(index=False))

    print("\nCOLUMN SUMMARY (sample only):")
    summary = pd.DataFrame({
        "data_type": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing_percent": df.isna().mean().mul(100).round(2),
        "unique_values": df.nunique(),
    })
    print(summary.to_string())

Total datasets: 42

DATASET: AR_merged.csv
Columns: 14
Rows inspected: 1,000 (maximum 1,000)

FIRST 5 ROWS:
        raw_row_number       date     time  lat  lng  subject_age subject_race subject_sex      type  citation_issued  outcome           vehicle_type raw_defendant_race                       source_file
            8620|12751 2017-01-01 01:50:15  NaN  NaN         55.0        white        male vehicular             True citation                  Other          CAUCASIAN ar_little_rock_2020_04_01.csv.zip
5296|10153|13440|14777 2017-01-01 05:24:24  NaN  NaN         52.0        black      female vehicular             True citation          Passenger Car   AFRICAN AMERICAN ar_little_rock_2020_04_01.csv.zip
                 16622 2017-01-01 05:56:20  NaN  NaN         35.0        black        male vehicular             True citation          Passenger Car   AFRICAN AMERICAN ar_little_rock_2020_04_01.csv.zip
                  4386 2017-01-01 06:35:46  NaN  NaN         20.0        black  

In [5]:
# Replace CA with the state you want.
file = data_folder / "MN_merged.csv"
df = pd.read_csv(file, nrows=1000, low_memory=False)

display(df.head(10))
display(df.describe(include="all").T)
print(df.columns.tolist())

,raw_row_number,date,time,lat,lng,police_grid_number,subject_age,subject_race,subject_sex,type,citation_issued,outcome,frisk_performed,search_conducted,search_vehicle,raw_race_of_driver,source_file
0,93569,2001-01-01,01:13:00,44.981102,-93.020193,39,NaN,white,male,vehicular,False,NaN,False,False,False,White,mn_saint_paul_2020_04_01.csv.zip
1,93570,2001-01-01,01:14:00,44.959361,-93.141547,87,NaN,black,male,vehicular,False,NaN,True,False,False,Black,mn_saint_paul_2020_04_01.csv.zip
2,93571,2001-01-01,01:47:00,44.959163,-93.071815,94,NaN,hispanic,male,vehicular,False,NaN,False,False,False,Hispanic,mn_saint_paul_2020_04_01.csv.zip
3,93572,2001-01-01,02:22:00,44.960154,-93.091979,92,NaN,asian/pacific islander,male,vehicular,False,NaN,True,True,True,Asian,mn_saint_paul_2020_04_01.csv.zip
4,93581,2001-01-01,15:01:00,44.959361,-93.141547,87,NaN,black,male,vehicular,False,NaN,True,True,True,Black,mn_saint_paul_2020_04_01.csv.zip
5,93573,2001-01-01,03:04:00,44.959294,-93.161887,85,NaN,white,male,vehicular,False,NaN,True,True,True,White,mn_saint_paul_2020_04_01.csv.zip
6,93574,2001-01-01,03:10:00,44.952118,-93.151733,106,NaN,asian/pacific islander,male,vehicular,False,NaN,False,False,False,Asian,mn_saint_paul_2020_04_01.csv.zip
7,93582,2001-01-01,15:22:00,44.973534,-93.092509,52,NaN,black,male,vehicular,False,NaN,False,False,False,Black,mn_saint_paul_2020_04_01.csv.zip
8,93575,2001-01-01,03:30:00,44.984757,-93.202710,1,NaN,black,male,vehicular,False,NaN,False,False,False,Black,mn_saint_paul_2020_04_01.csv.zip
9,93583,2001-01-01,15:33:00,44.959340,-93.131400,88,NaN,black,female,vehicular,False,NaN,False,False,False,Black,mn_saint_paul_2020_04_01.csv.zip


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
raw_row_number,1000.0,NaN,NaN,NaN,317694.171,205102.526625,1717.0,93677.75,321993.5,507473.25,657211.0
date,1000,17,2004-01-01,107,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,1000,593,23:31:00,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1000.0,NaN,NaN,NaN,44.959436,0.017285,44.898384,44.952065,44.959798,44.973748,44.988419
lng,1000.0,NaN,NaN,NaN,-93.101325,0.041502,-93.204229,-93.1314,-93.101174,-93.071036,-93.010104
police_grid_number,1000.0,NaN,NaN,NaN,94.905,52.538179,1.0,55.0,89.0,112.0,269.0
subject_age,136.0,NaN,NaN,NaN,31.639706,12.139725,14.0,22.0,27.5,37.25,69.0
subject_race,874,5,white,357,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subject_sex,876,2,male,602,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,1000,1,vehicular,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['raw_row_number', 'date', 'time', 'lat', 'lng', 'police_grid_number', 'subject_age', 'subject_race', 'subject_sex', 'type', 'citation_issued', 'outcome', 'frisk_performed', 'search_conducted', 'search_vehicle', 'raw_race_of_driver', 'source_file']


In [2]:
# Replace CA with the state you want.
file = Documents/GitHub/UVD-Project-/data/merged_by_state/AZ_merged.csv"
df = pd.read_csv(file, nrows=1000, low_memory=False)

display(df.head(10))
display(df.describe(include="all").T)
print(df.columns.tolist())

SyntaxError: unterminated string literal (detected at line 2) (2556754946.py, line 2)

In [3]:
import pandas as pd
from pathlib import Path

# Replace AZ with the state you want.
file = Path.home() / "Documents/GitHub/UVD-Project-/data/merged_by_state/AZ_merged.csv"
df = pd.read_csv(file, nrows=1000, low_memory=False)

display(df.head(10))
display(df.describe(include="all").T)
print(df.columns.tolist())

,raw_row_number,date,time,location,lat,lng,officer_id_hash,type,vehicle_color,vehicle_make,...,search_vehicle,search_basis,reason_for_stop,vehicle_type,raw_Ethnicity,raw_OutcomeOfStop,raw_ReasonForStop,raw_TypeOfSearch,raw_ViolationsObserved,source_file
0,1,2008-01-01,00:02:00,S POWER RD / E QUEEN CREEK RD,33.266353,-111.686102,52e00e574e,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
1,2,2008-01-01,00:08:00,APACHE / ELM,33.360515,-111.787292,442db3e402,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
2,3,2008-01-01,00:13:00,W GUADALUPE RD / N VELERO ST,33.364319,-111.811403,044611abc4,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
3,4|5|6|7|8,2008-01-01,00:18:00,BROADWAY / RIVER,33.352826,-111.789027,17e9490975,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
4,9,2008-01-01,00:23:00,N FIESTA BLVD / W GUADALUPE RD,33.364226,-111.834276,044611abc4,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
5,10|11|12|13,2008-01-01,00:30:00,N GRANITE ST / N STONECREEK BLVD,33.358123,-111.760274,62447ded99,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
6,14,2008-01-01,00:31:00,W ELLIOT RD / N MCQUEEN RD,33.349818,-111.824287,3ea44364b0,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
7,15|16|17,2008-01-01,00:39:00,N GILBERT RD / E PAGE AVE,33.354870,-111.789717,9a2227c1d6,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
8,18,2008-01-01,00:49:00,E GUADALUPE RD / N RIATA ST,33.364551,-111.777722,044611abc4,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip
9,19,2008-01-01,00:49:00,E PALO VERDE ST / S VAL VISTA DR,33.346543,-111.755607,540f9fc4cc,vehicular,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az_gilbert_2020_04_01.csv.zip


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
raw_row_number,1000,1000,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,1000,8,2008-01-03,184,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time,1000,699,23:27:00,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,1000,421,S GREENFIELD RD / E PECOS RD,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1000.0,NaN,NaN,NaN,33.326523,0.03882,33.212106,33.292333,33.335289,33.361476,33.483996
lng,1000.0,NaN,NaN,NaN,-111.752974,0.03854,-111.928958,-111.782293,-111.755409,-111.722304,-111.511365
officer_id_hash,1000,131,5ab05d03c4,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,1000,1,vehicular,1000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_color,1,1,GRY,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_make,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['raw_row_number', 'date', 'time', 'location', 'lat', 'lng', 'officer_id_hash', 'type', 'vehicle_color', 'vehicle_make', 'vehicle_model', 'vehicle_year', 'geocode_source', 'subject_age', 'subject_race', 'subject_sex', 'violation', 'arrest_made', 'citation_issued', 'warning_issued', 'outcome', 'raw_race_fixed', 'raw_ethnicity_fixed', 'raw_charge', 'county_name', 'contraband_found', 'contraband_drugs', 'contraband_other', 'search_conducted', 'search_person', 'search_vehicle', 'search_basis', 'reason_for_stop', 'vehicle_type', 'raw_Ethnicity', 'raw_OutcomeOfStop', 'raw_ReasonForStop', 'raw_TypeOfSearch', 'raw_ViolationsObserved', 'source_file']
